In [115]:
from nrem_analysis.constant import PROCESSED_DIR, INTERIM_DIR
from pathlib import Path

import numpy as np
import pynapple as nap
import matplotlib.pyplot as plt

from cmap import Colormap
from matplotlib.colors import TwoSlopeNorm

cm = Colormap("colorbrewer:Set2")
norm = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

plt.rcParams.update({
    "figure.dpi":        150,
    "savefig.dpi":       320,
    "font.family":       "sans-serif",
    'font.sans-serif':   "Arial",
    "axes.grid":         False,
    "figure.constrained_layout.use": True,
})

AWAKE_STA_KWARGS = dict(binsize=0.05, window=(-2.025, 2.025))
NREM_STA_KWARGS = dict(binsize=0.005, window=(-0.4, 0.4))

STATE_NAMES = ["continuous", "fragmented", "stationary"]
MOUSE_IDS = ["99b", "103c", "106b", "107b", "110b"]

def compute_unwrapped_sta(data, events, window, binsize):
    perievent = nap.compute_perievent(data=data, events=events, window=window,)
    # unwrap each sweep
    result = np.unwrap(perievent, axis=0)
    result = np.nanmean(result, axis=1)
    zero_idx = np.argmin(np.abs(result.index))
    result = result - result[zero_idx]
    return result.bin_average(bin_size=binsize)

In [ ]:
tsi = []
sta_awake = []
sta_nrem = []

for mouse_id in MOUSE_IDS:
    turn_units      = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "turn_units.npz")
    head_direction  = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "head_direction.npz")
    sweeps          = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "sweeps.npz")

    head_direction = np.deg2rad(head_direction)
    sweeps = np.deg2rad(sweeps)

    for uid in turn_units.index:
        for data, container, kwargs in zip([head_direction, sweeps], [sta_awake, sta_nrem], [AWAKE_STA_KWARGS, NREM_STA_KWARGS],):
            spikes_in_event = turn_units[uid].restrict(data.time_support)
            if spikes_in_event:
                sta_per_unit = compute_unwrapped_sta(data, spikes_in_event, **kwargs)

                tsi.append(turn_units['turn_index'][uid])
                container.append(sta_per_unit)
            else:
                print(f"Skipping {uid}")

tsi = np.concatenate(tsi)
sta_awake = np.column_stack(sta_awake)
sta_nrem = np.column_stack(sta_nrem)

/Users/iii9781/nrem_analysis/.venv/lib/python3.13/site-packages/pynapple/core/time_series.py:329: RuntimeWarning: Mean of empty slice
  out = func._implementation(*new_args, **kwargs)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
axes = axes.flatten()

for i, (sta, state) in enumerate(zip([sta_awake, sta_nrem], ["Awake", "NREM"])):
    im = axes[i].imshow(sta.d.T, aspect='auto', origin='lower',
                        extent=[sta.index[0], sta.index[-1], 0, sta.shape[1]],
                        cmap='hot',
                        # norm=norm,
                        )
    
    plt.colorbar(im, ax=axes[i], label='Rel. HD (deg)')
    axes[i].set_title(f"{state}", fontsize=22)
    axes[i].set_xlabel("Time from spike [s]")
    
axes[0].set_ylabel("Turn Cell ID")
axes[0].set_yticks([0, sta.shape[1]-1])
# axes[0].set_xticks([-0.5, 0, 0.5])
# # axes[1].set_xticks([-0.2, 0, 0.2])
# plt.savefig(FIGURES_DIR / "sta" /  "mv2st99" / "heatmap_turn.png")